In [ ]:
# ============================================================
# 02 Baseline Random Forest Results Analysis
# ============================================================

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_curve,
    precision_recall_curve,
    auc,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize

from lif_thesis.models.baseline_rf import build_spectra_matrix

In [ ]:
# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = Path.cwd().parents[1]

DATA_PATH = ROOT / "data" / "processed" / "bacterial_samples.parquet"
RESULTS_DIR = ROOT / "results" / "exp00_paper_reproduction"

METRICS_PATH = RESULTS_DIR / "metrics.json"
MODEL_PATH = RESULTS_DIR / "exp00_paper_reproduction.joblib"
ENCODER_PATH = RESULTS_DIR / "label_encoder.joblib"

print("Root:", ROOT)
print("Data path:", DATA_PATH)
print("Results path:", RESULTS_DIR)

In [ ]:
# ------------------------------------------------------------
# 2. Load data, model, metrics, and split indices
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

model = joblib.load(MODEL_PATH)
label_encoder = joblib.load(ENCODER_PATH)

with open(METRICS_PATH, "r") as f:
    metrics = json.load(f)

train_idx = np.load(RESULTS_DIR / "train_idx.npy")
val_idx = np.load(RESULTS_DIR / "val_idx.npy")
test_idx = np.load(RESULTS_DIR / "test_idx.npy")

print("Data shape:", df.shape)
print("Classes:", label_encoder.classes_)

In [ ]:
# ------------------------------------------------------------
# 3. Summarize split composition
# ------------------------------------------------------------

split_summary = []

for split_name, idx in {
    "train": train_idx,
    "validation": val_idx,
    "test": test_idx,
}.items():
    split_df = df.iloc[idx]

    split_summary.append({
        "split": split_name,
        "n_particles": len(split_df),
        "n_raw_files": split_df["raw_file"].nunique(),
        "n_species": split_df["species"].nunique(),
    })

split_summary_df = pd.DataFrame(split_summary)
display(split_summary_df)

for split_name, idx in {
    "train": train_idx,
    "validation": val_idx,
    "test": test_idx,
}.items():
    print(f"\n{split_name.upper()} species distribution")
    display(df.iloc[idx]["species"].value_counts(normalize=True))